In [1]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

#dataset_path = "/slowdisk/data/DAC/pitchglidesATriangle5octaves/Aglides" # Path to your dataset directory
dataset_path = "/slowdisk/data/DAC/waterfill" # Path to your dataset directory
            

In [2]:
repo_root

PosixPath('/home/lonce/working/agent_projects/FargoNDacoder')

# **Dataset Maker**

Follow this notebook to transform your data into a dataset that can be used to train the model.

## 📋 **1. Required Data**

FARGO requires three types of data: **audio files**, **annotations**, and **parameter specifications**. Each one is explained below.

### **1.1 🎵 Audio Files**

Most audio file formats and sampling rates are supported, although the final model will always operate at 24kHz.

**Supported Audio Formats**: `.wav`, `.mp3`, `.flac`, `.aac`, `.ogg`, `.m4a`, `.wma`, `.aiff`, `.au`, `.ra`, `.3gp`, `.amr`, `.ac3`, `.dts`, `.ape`, `.mka`, `.opus`

### **1.2 📊 Annotations (CSV Files)**

Each audio file **must** have a corresponding CSV file with the **exact same base name**:

| Audio File | CSV File | Status |
|------------|----------|---------|
| `piano.wav` | `piano.csv` | ✅ Correct |
| `flute.mp3` | `flute_annotations.csv` | ⛔ Incorrect - name mismatch! |

The CSV annotation files must follow this specific structure:

- **Header row** with parameter names (must match those in `parameters.json`)
- **One row per frame** (86.13 frames per second)
- **Continuous parameters**: Numeric values (will be normalized to [0,1] range)
- **Categorical parameters**: Non-negative integers representing each class (0, 1, 2, ...)

Example CSV:

| saturation | reverb | instrument |
|------------|--------|------------|
| 10.5       | 40.3   | 0          |
| 10.0       | 40.32  | 1          |
| 9.8        | 40.25  | 0          |
| ...        | ...    | ...        |

### **1.3 ⚙️ Parameter Specifications**

Information about the parameters to be controlled must be stored in a `parameters.json` file. Each parameter must specify its **name** and **type** (either `continuous` or `class`). Additional features will depepnd on the type of the parameter as follow:

**🔢 Continuous Parameters** (numeric values like tempo, volume, saturation):
- `min`/`max`: Range used for normalization
- `unit`: Physical unit (e.g., bpm, %, dB)

**🏷️ Class Parameters** (categorical values like instrument type, genre):
- `classes`: Ordered list of class names (order determines integer mapping)

#### Example `parameters.json`:

```json
{
    "parameter_1": {"name": "saturation", "type": "continuous", "unit": "dB", "min": 0,   "max": 12}, 
    "parameter_2": {"name": "reverb",     "type": "continuous", "unit": "%",  "min": 0,   "max": 100},
    "parameter_3": {"name": "instrument", "type": "class",      "classes": ["piano", "flute"]}
}
```
## **📁 2. Required Data Structure**

Before running this notebook on your data, make sure it is structured as follows:

**Option 1:** Simple Structure (all data together)

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── piano.wav                # Audio files (various formats supported)
    ├── piano.csv                # Corresponding CSV annotations
    ├── flute.mp3                # More audio files...
    ├── flute.csv                # More CSV files...
    └── ...
```

**Option 2:** Split Structure (separate train/validation/test sets)
If you want to use specific data splits for training, validation, and testing.

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── train/
    │   ├── piano.wav            # Training audio files
    │   ├── piano.csv            # Training CSV annotations
    │   └── ...
    ├── validation/
    │   ├── flute.mp3            # Validation audio files
    │   ├── flute.csv            # Validation CSV files
    │   └── ...
    └── test/
        ├── violin.mp3           # Test audio files
        ├── violin.csv           # Test CSV files
        └── ...
```

## **🔄 3. Dataset Creation Pipeline**

Once your data is properly arranged, follow this notebook to create your dataset. The full pipeline consists of five steps:

- **📊 Visualization** - Analyze your raw data structure and parameters to verify everything is in order
- **🔧 Normalization** - Standardize format and normalize your data
- **🎵 DAC Encoding** - Convert audio to a compressed latent format that FARGO can process
- **📄 Sidecar Creation**- Align parameters with audio frames (86.13 fps)
- **🤗 HuggingFace Dataset** - Package everything into a training-ready format

Let's get started! 👇

### **3.1 📊 Visualization**

Visualize your data and make sure everything is alright.

In [3]:
# Import visualization functions
from dataprep.step_0_visualization_dac import quick_analyze

# 🔍 ANALYZE ENTIRE DATASET
quick_analyze(dataset_path, individual_plot_selector=True) # Set individual_plot_selector=True to choose which files to plot.

DATASET SUMMARY (DAC / 44.1 kHz):

✅ Found 28 audio-CSV pairs
📁 Parameters: fill_level
⏱️ Total audio duration: 404.0 seconds
🎛️ DAC sample rate: 44100 Hz
🎛️ DAC hop length: 512 samples
🎛️ DAC frame rate: 86.132812 fps

FILE DETAILS:

≈ ZOOM0026: 15.77s, CSV rows=1359, target DAC frames=1359, CSV fps≈86.181
≈ ZOOM0027: 16.18s, CSV rows=1394, target DAC frames=1394, CSV fps≈86.134
≈ ZOOM0028: 14.46s, CSV rows=1246, target DAC frames=1246, CSV fps≈86.145
≈ ZOOM0001: 14.03s, CSV rows=1209, target DAC frames=1209, CSV fps≈86.193
≈ ZOOM0002: 13.89s, CSV rows=1197, target DAC frames=1197, CSV fps≈86.170
≈ ZOOM0003: 13.25s, CSV rows=1142, target DAC frames=1142, CSV fps≈86.174
≈ ZOOM0004: 12.87s, CSV rows=1109, target DAC frames=1109, CSV fps≈86.175
≈ ZOOM0005: 15.16s, CSV rows=1307, target DAC frames=1307, CSV fps≈86.194
≈ ZOOM0006: 14.77s, CSV rows=1272, target DAC frames=1272, CSV fps≈86.146
≈ ZOOM0007: 14.45s, CSV rows=1245, target DAC frames=1245, CSV fps≈86.171
≈ ZOOM0008: 14.69s, CSV r

### **3.2 🔧 Normalization**

This step prepares your audio files by:
1. **Resampling** all audio to 44.1kHz mono (required by DAC)
2. **RMS Normalization** (optional) to ensure consistent loudness across your dataset (you may want apply_rms_normalization=False if the relative volume between your dataset files is important).

In [4]:
from dataprep.step_1_normalization_dac import quick_normalize

quick_normalize(dataset_path, apply_rms_normalization=False)


DATA SUMMARY:

Found 28 audio-CSV pairs
Target sample rate: 44100 Hz mono
RMS normalization: DISABLED
Input:  /slowdisk/data/DAC/waterfill/raw
Output: /slowdisk/data/DAC/waterfill/normalized

DATA PROCESSING:

Processing: ZOOM0026.wav
    RMS normalization disabled - resampling to 44100 Hz only
    ✓ Saved: ZOOM0026.wav
    ✅ Copied CSV: ZOOM0026.csv → /slowdisk/data/DAC/waterfill/normalized/test/ZOOM0026.csv

Processing: ZOOM0027.wav
    RMS normalization disabled - resampling to 44100 Hz only
    ✓ Saved: ZOOM0027.wav
    ✅ Copied CSV: ZOOM0027.csv → /slowdisk/data/DAC/waterfill/normalized/test/ZOOM0027.csv

Processing: ZOOM0028.wav
    RMS normalization disabled - resampling to 44100 Hz only
    ✓ Saved: ZOOM0028.wav
    ✅ Copied CSV: ZOOM0028.csv → /slowdisk/data/DAC/waterfill/normalized/test/ZOOM0028.csv

Processing: ZOOM0001.wav
    RMS normalization disabled - resampling to 44100 Hz only
    ✓ Saved: ZOOM0001.wav
    ✅ Copied CSV: ZOOM0001.csv → /slowdisk/data/DAC/waterfill/nor

### **3.3 🎵 EDAC Encoding**

In [5]:
from dataprep.step_2_dac import quick_encode

quick_encode(dataset_path, device="cpu")  # Force 8 codebooks and CPU


DATA PROCESSING:

Found 28 audio files under /slowdisk/data/DAC/waterfill/normalized
Using device: cpu
DAC model: 44khz
win_duration: None
normalize_db: None
n_quantizers: 9


/home/lonce/miniconda3/envs/synthformer/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Encoding: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 28/28 [01:29<00:00,  3.20s/file]


{'total': 28, 'success': 28, 'skipped': 0, 'failed': 0, 'errors': []}

### **3.4 📄 Sidecar Creation**

In [6]:
from dataprep.step_3_sidecars_dac import quick_create_sidecars

results = quick_create_sidecars(dataset_path)


SIDECAR CREATION:

📁 Tokens directory: /slowdisk/data/DAC/waterfill/tokens
📁 Raw directory:    /slowdisk/data/DAC/waterfill/raw
📊 Found 28 .dac-CSV pairs
🎛️ Parameters:      fill_level

SIDECARS SUMMARY:

📄 ZOOM0025 sidecar:
    DAC frames: 1256
    CSV rows:   1256
    DAC fps:    86.132812
    ✅ Sidecar created (1256 frames, 1 features)

📄 ZOOM0003 sidecar:
    DAC frames: 1142
    CSV rows:   1142
    DAC fps:    86.132812
    ✅ Sidecar created (1142 frames, 1 features)

📄 ZOOM0015 sidecar:
    DAC frames: 1237
    CSV rows:   1237
    DAC fps:    86.132812
    ✅ Sidecar created (1237 frames, 1 features)

📄 ZOOM0019 sidecar:
    DAC frames: 1223
    CSV rows:   1223
    DAC fps:    86.132812
    ✅ Sidecar created (1223 frames, 1 features)

📄 ZOOM0013 sidecar:
    DAC frames: 1120
    CSV rows:   1120
    DAC fps:    86.132812
    ✅ Sidecar created (1120 frames, 1 features)

📄 ZOOM0006 sidecar:
    DAC frames: 1272
    CSV rows:   1272
    DAC fps:    86.132812
    ✅ Sidecar created

### **3.5 🤗 HuggingFace Dataset**
(on Windows, you can ignore the "Failed simlink" messages)

In [7]:
from dataprep.step_4_HF_dac import quick_create_dataset

results = quick_create_dataset(dataset_path)


HUGGINGFACE DATASET CREATION:

📁 Tokens directory: /slowdisk/data/DAC/waterfill/tokens
📁 Output directory: /slowdisk/data/DAC/waterfill/hf_dataset
🔗 Materialize mode: link

🎯 Split structure detected: train, test
   📂 train: 25 .dac files found
   📂 test: 3 .dac files found

📋 Saved expanded conditioning config to: conditioning_config.json
   • Features: fill_level
   • Total features: 1

💾 Saved DatasetDict to: /slowdisk/data/DAC/waterfill/hf_dataset
📈 Dataset summary:
   • train: 25 samples
   • test: 3 samples

🔍 Verifying dataset files...
✅ All DAC files verified successfully
